# A Latent Diffusion model combining ideas from CS336's language model and a couple image generation papers

Based on:
* The language transformer from CS336 (multihead self attention, adamw, swiglu, rope)
* X-prediction and flow matching from [Let Denoising Generative Models Denoise (Tianhong Li, Kaiming He, et al., 2025)](https://arxiv.org/abs/2511.13720)
* A Vector Quantized latent space from ["Generative Refinement Networks for Visual Synthesis", 
(Han et al., 2026)](https://arxiv.org/abs/2604.13030)

* https://claude.ai/share/9c11840b-4ad9-4736-8119-47e2060b7a02 - GRN -> JiT
* https://claude.ai/chat/4b918b44-b26c-445f-ac9d-7899e60f6875 - Rope based autoencoder vs limited receptive field

In [ ]:
#!uv sync --group=dev
#!uv pip install https://github.com/ramayer/cs336-assignment1-basics.git
#!uv pip install https://github.com/ramayer/hierarchical-binary-quantization.git

In [ ]:
import einx
import datetime
import IPython.display as ipd
import math
import numpy as np
import time
import torch
import torch.nn as nn
from pathlib import Path
from torch.utils.data import Dataset, DataLoader

import cs336_basics.ron_adamw_optimizer as cs336_adamw_optimizer
import cs336_basics.ron_bpe_tokenizer as cs336__bpe_tokenizer
import cs336_basics.ron_causal_multihead_self_attention_with_rope as cs336_causal_multihead_self_attention_with_rope
import cs336_basics.ron_cross_entropy as cs336_cross_entropy
import cs336_basics.ron_data_loader as cs336_data_loader
import cs336_basics.ron_embedding as cs336_embedding
import cs336_basics.ron_linear as cs336_linear
import cs336_basics.ron_multihead_self_attention as cs336_multihead_self_attention
import cs336_basics.ron_rmsnorm as cs336_rmsnorm
import cs336_basics.ron_rope as cs336_rope
import cs336_basics.ron_scaled_dot_product_attention as cs336_scaled_dot_product_attention
import cs336_basics.ron_softmax as cs336_softmax
import cs336_basics.ron_swiglu as cs336_swiglu
import cs336_basics.ron_train_bpe as cs336_train_bpe
import cs336_basics.ron_transformer_lm as cs336_transformer_lm

import hierarchical_binary_quantization.hbq as hbq
import hierarchical_binary_quantization.misc as hbq_misc 
import hierarchical_binary_quantization.misc.image_helpers as ih
import hierarchical_binary_quantization.misc.mlflow_helper as mh
from hierarchical_binary_quantization.example_autoencoder_with_rope import ExampleQuantizingAutoencoderWithRope


device='cuda'


In [ ]:
height,width=384,384
grid_size=height//8

good_autoencoder_checkpoint = '../assets/reallygood_LocalQuantizingAutoencoder_2026-09-12_07-44-00_ps8,qd16,ld32,body:res,attn,res.pth'
good_latent_xpred_diffusion_cp =  "checkpoints/nice_2026-09-13_19-48-58_fantasy_cs336_latent_jit.pth_ema.pth"


## Torch module variations to convert the CS336 language model to an Image Model

### Rope2D.  Like CS336's 1D one, but in 2D.

In [ ]:
class RoPE2D(torch.nn.Module):
    def __init__(self, theta: float, d_k: int, grid_size: int, device=None):
        super().__init__()
        assert d_k % 4 == 0, "need d_k/2 even, since each half gets your original RoPE"
        self.rope_row = cs336_rope.RoPE(theta=theta, d_k=d_k // 2, max_seq_len=grid_size, device=device)
        self.rope_col = cs336_rope.RoPE(theta=theta, d_k=d_k // 2, max_seq_len=grid_size, device=device)

    def forward(self, x, rows, cols):
        """
        x: (..., seq_len, d_k)
        rows, cols: (seq_len,) -- from get_2d_positions()
        """
        d_half = x.shape[-1] // 2
        x_row_half, x_col_half = x[..., :d_half], x[..., d_half:]
        x_row_half = self.rope_row(x_row_half, token_positions=rows)
        x_col_half = self.rope_col(x_col_half, token_positions=cols)
        return torch.cat([x_row_half, x_col_half], dim=-1)

def get_2d_positions(grid_size):
    """
    For a grid_size x grid_size grid, returns (rows, cols) — each shape
    (grid_size*grid_size,) — giving every flattened token's row and column.

    Token order matches how we'll flatten the latent tensor: x.flatten(2)
    on a (C, H, W) tensor walks W fastest, then H — i.e. row 0's columns
    0..31, then row 1's columns 0..31, etc. So token index t corresponds
    to row = t // grid_size, col = t % grid_size.
    """
    rows = torch.arange(grid_size).unsqueeze(1).expand(grid_size, grid_size).reshape(-1)
    cols = torch.arange(grid_size).unsqueeze(0).expand(grid_size, grid_size).reshape(-1)
    return rows, cols


def smoke_test_rope():
    one_d_rope = cs336_rope.RoPE(theta=150,d_k=4,max_seq_len=4)(torch.ones(4),token_positions=None)
    print("1D rope",one_d_rope)
    grid_size = 2
    rows, cols = get_2d_positions(grid_size)
    print("rows ",rows)
    print("cols ",cols)
    two_d_rope = RoPE2D(150, 4, grid_size)(torch.ones(4, 4), rows, cols)
    print("2D rope", two_d_rope)
smoke_test_rope()


### 2D MultiheadSelfAttention. Like CS336's 1D one, but 2D

In [ ]:


class SpatialMultiheadSelfAttention(torch.nn.Module):
    def __init__(self, d_model, num_heads, theta, grid_size):
        super().__init__()
        dk = dv = d_model // num_heads
        self.num_heads = num_heads
        self.q_proj = nn.Linear(d_model, num_heads * dk)
        self.k_proj = nn.Linear(d_model, num_heads * dk)
        self.v_proj = nn.Linear(d_model, num_heads * dv)
        self.o_proj = nn.Linear(num_heads * dv, d_model)
        self.rope2d = RoPE2D(d_k=dk, grid_size=grid_size, theta=theta)

    def forward(self, x, rows, cols):
        q, k, v = self.q_proj(x), self.k_proj(x), self.v_proj(x)
        q = einx.id("... seq_len (heads d) -> ... heads seq_len d", q, heads=self.num_heads)
        k = einx.id("... seq_len (heads d) -> ... heads seq_len d", k, heads=self.num_heads)
        v = einx.id("... seq_len (heads d) -> ... heads seq_len d", v, heads=self.num_heads)

        # broadcast rows/cols across the heads dim, same trick as the original's token_positions
        heads_rows = einx.id("... s -> ... 1 s", rows)
        heads_cols = einx.id("... s -> ... 1 s", cols)
        q = self.rope2d(q, rows=heads_rows, cols=heads_cols)
        k = self.rope2d(k, rows=heads_rows, cols=heads_cols)

        # no mask at all -- every latent-grid token attends to every other token
        #attn_output = torch.nn.functional.scaled_dot_product_attention(q, k, v, attn_mask=None)
        attn_output = cs336_scaled_dot_product_attention.scaled_dot_product_attention(k=k, q=q, v=v, mask=None)
        attn_output = einx.id("... heads seq d_v -> ... seq (heads d_v)", attn_output)
        return self.o_proj(attn_output)


def smoke_test_SpatialMultiheadSelfAttention():
    rows, cols = get_2d_positions(4)
    smsa = SpatialMultiheadSelfAttention(d_model=384, num_heads=4, theta=1000, grid_size=4)
    print(smsa.forward(torch.randn(2, 16, 384), rows, cols)[0:1,0:4,0:4])

smoke_test_SpatialMultiheadSelfAttention()


### 2D TransformerBlock. Like CS336's 1D one, but 2D.

* Like CS336's 1D one, but 2D

In [ ]:
class SpatialTransformerBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, theta, grid_size):
        super().__init__()
        self.norm1 = cs336_rmsnorm.RMSNorm(d_model)
        self.attn = SpatialMultiheadSelfAttention(d_model=d_model, num_heads=num_heads,
                                                   theta=theta, grid_size=grid_size)
        self.norm2 = cs336_rmsnorm.RMSNorm(d_model)
        self.ffn = cs336_swiglu.SwiGLU(d_model, d_ff)

    def forward(self, x, rows, cols):
        x = x + self.attn(self.norm1(x), rows, cols)
        x = x + self.ffn(self.norm2(x))
        return x

def timestep_embedding(t, dim):
    """Standard sinusoidal embedding, same idea as positional encodings, applied to a scalar 't' instead of a sequence index."""
    half = dim // 2
    freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=-1)  # (batch, dim)

### AdaLN - using CS336 components

* AdaLN is a standard way of doing time conditioning on diffusion models, but buid this one out of CS336's SwiGLU and RMSNorm

In [ ]:

def modulate(x, shift, scale):
    """
    x: (B, seq_len, d_model)
    shift, scale: (B, d_model) -- one value per channel, per batch item, from t
    """
    return x * (1 + scale.unsqueeze(1)) + shift.unsqueeze(1)

class AdaLNSpatialTransformerBlock(torch.nn.Module):
    def __init__(self, d_model, num_heads, d_ff, theta, grid_size):
        super().__init__()
        self.norm1 = cs336_rmsnorm.RMSNorm(d_model)
        self.attn = SpatialMultiheadSelfAttention(d_model=d_model, num_heads=num_heads,
                                                   theta=theta, grid_size=grid_size)
        self.norm2 = cs336_rmsnorm.RMSNorm(d_model)
        self.ffn = cs336_swiglu.SwiGLU(d_model, d_ff)

        self.adaLN_modulation = torch.nn.Linear(d_model, 6 * d_model)
        torch.nn.init.zeros_(self.adaLN_modulation.weight)
        torch.nn.init.zeros_(self.adaLN_modulation.bias)

    def forward(self, x, rows, cols, t_emb):
        shift_attn, scale_attn, gate_attn, shift_ffn, scale_ffn, gate_ffn = \
            self.adaLN_modulation(t_emb).chunk(6, dim=-1)

        attn_in = modulate(self.norm1(x), shift_attn, scale_attn)
        x = x + gate_attn.unsqueeze(1) * self.attn(attn_in, rows, cols)

        ffn_in = modulate(self.norm2(x), shift_ffn, scale_ffn)
        x = x + gate_ffn.unsqueeze(1) * self.ffn(ffn_in)

        return x

def smoke_test_AdaLNSpatialTransformerBlock():
    batch_size = 7
    d_model=100
    num_heads=5
    d_ff=17
    theta=1111
    grid_size = 3

    alnstb = AdaLNSpatialTransformerBlock(d_model, num_heads, d_ff, theta, grid_size)
    rows, cols = get_2d_positions(grid_size)
    x = torch.rand(batch_size,grid_size*grid_size, d_model)
    t = torch.rand(batch_size)
    t_emb = timestep_embedding(t, d_model)  
    out = alnstb.forward(x,rows,cols, t_emb)
    print(torch.allclose(out, x))        # should be True -- zero-init means no-op at init
smoke_test_AdaLNSpatialTransformerBlock()

### Minimal Vision Transformer using CS336 blocks.

Just like the LLM, except:

* Input tokens come from a quantizing autoencoder's encoder that preprocessed the images instead of BPE.
* Outputs are tokens for the quantizing auotencoder's decoder instead of BPE tokens.

In [ ]:
class MinimalSpatialViT(torch.nn.Module):
    def __init__(self, grid_size=48, latent_dim=8, d_model=384, num_heads=6,
                 d_ff=1024, depth=6, theta=150.0):
        super().__init__()
        self.grid_size = grid_size
        self.latent_dim = latent_dim
        self.d_model = d_model
        self.x_embedder = cs336_linear.Linear(latent_dim, d_model)
        self.blocks = torch.nn.ModuleList([
            AdaLNSpatialTransformerBlock(d_model, num_heads, d_ff, theta, grid_size)
            for _ in range(depth)
        ])
        self.final_norm = cs336_rmsnorm.RMSNorm(d_model)
        self.final_linear = cs336_linear.Linear(d_model, latent_dim)
        rows, cols = get_2d_positions(grid_size)
        self.register_buffer("rows", rows)
        self.register_buffer("cols", cols)
        self.config = {
            "grid_size":grid_size,
            "latent_dim":latent_dim,
            "d_model":d_model,
            "num_heads":num_heads,
            "d_ff":d_ff,
            "depth":depth,
            "theta":theta,
        }

    def forward(self, x, t, y=None):
        B, C, H, W = x.shape
        tokens = x.flatten(2).transpose(1, 2)      # (B, H*W, latent_dim)
        tokens = self.x_embedder(tokens)             # (B, H*W, d_model)
        t_emb = timestep_embedding(t, self.d_model)   # (B, d_model)
        for block in self.blocks:
            tokens = block(tokens, self.rows, self.cols, t_emb)
        tokens = self.final_linear(self.final_norm(tokens))
        return tokens.transpose(1, 2).reshape(B, C, H, W)

def smoke_test_MinimalSpatialViT(grid_size=48):
    BS = 3
    lj = MinimalSpatialViT(grid_size=48).to(device)
    total_params = sum(p.numel() for p in lj.parameters())
    print(f"Total parameters: {total_params:,}")
    x = torch.randn(BS, 8, grid_size,grid_size).to(device)
    t = torch.zeros(BS, dtype=torch.long).to(device)
    labels = torch.zeros(BS, dtype=torch.long).to(device)
    print("output: ",lj.forward(x,t,labels).shape)

smoke_test_MinimalSpatialViT(grid_size=48)

In [ ]:
def smoke_test_MinimalSpatialViT():
    BS = 3
    lj = MinimalSpatialViT(grid_size=grid_size).to(device)
    total_params = sum(p.numel() for p in lj.parameters())
    print(f"Total parameters: {total_params:,}")
    x = torch.randn(BS, 8, grid_size,grid_size).to(device)
    t = torch.zeros(BS, dtype=torch.long).to(device)
    labels = torch.zeros(BS, dtype=torch.long).to(device)
    print("output: ",lj.forward(x,t,labels).shape)
smoke_test_MinimalSpatialViT()

## Load a quantizing autoencoder.

Could have been one from https://github.com/lucidrains/vector-quantize-pytorch, but I prefer my own

In [ ]:
#checkpoint_path = '../../hierarchical-binary-quantization/notebooks/checkpoints/ExampleQuantizingAutoencoder_2026-08-15_12-48-36_.pth'
#autoencoder = hbq_misc.checkpoint_helpers.create_from_checkpoint(ExampleQuantizingAutoencoderWithRope, good_autoencoder_checkpoint)
import hierarchical_binary_quantization.misc.checkpoint_helpers as ch
import hierarchical_binary_quantization.misc.dataset_helpers as dh

import hierarchical_binary_quantization.example_narrow_reciptive_field_autoencoder as enrfa
autoencoder = ch.create_from_checkpoint(enrfa.LocalQuantizingAutoencoder,good_autoencoder_checkpoint)
autoencoder.eval()
pass

## Load an image dataset

In [ ]:
# class LatentDataset(Dataset):
#     """Stage A: continuous (dequantized) latents at max rounds. No bits/rounds/masking yet."""
#     def __init__(self, root, max_rounds=4):
#         self.root = Path(root)
#         self.max_rounds = max_rounds
#         self.files = sorted(self.root.rglob("*.npz"))
#         print(f"Found {len(self.files)} cached latents.")

#     def __len__(self):
#         return len(self.files)

#     def __getitem__(self, idx):
#         data = np.load(self.files[idx])
#         bit_codes = torch.from_numpy(data["bit_codes"].astype(np.int64))  # (L, H, W)
#         bits = bit_codes.permute(1, 2, 0)                                  # (H, W, L)
#         latent = hbq.bit_codes_to_quantized_latent(bits, self.max_rounds)      # (H, W, L) floats
#         return latent.permute(2, 0, 1)  # (L, H, W) -- mirrors JiT's (C,H,W) image convention

# dataset='fantasy'

# def get_latent_dataset_and_dataloader(dataset,batch_size=8, shuffle=False):
#     if dataset == 'all':
#         dataset=''
#     latent_path=f'../../hierarchical-binary-quantization/notebooks/data/32x32x8latents_with_rope/{dataset}'
#     lds = LatentDataset(latent_path)
#     ldl = DataLoader(lds,shuffle=shuffle,batch_size=batch_size)
#     return lds,ldl


# def smoke_test_data_loader_and_autoencoder():
#     lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True)
#     ld_batch = next(iter(ldl))
#     print(f"lds shape {ld_batch.shape}")
#     ipd.display(hbq_misc.image_helpers.tensor_to_pil(autoencoder.backbone.decode(autoencoder.post_quant(ld_batch))[0]))

# smoke_test_data_loader_and_autoencoder()

In [ ]:
!ls -l data/dbs/*q*

In [ ]:
1

In [ ]:

# class LatentDataset(Dataset):
#     def __init__(self, blob_ds):
#         self.blob_ds = blob_ds
#     def __len__(self):
#         return len(self.blob_ds)
#     def __getitem__(self, idx):
#         id, data, md = self.blob_ds[idx]
#         bit_codes = dh.bytes_to_tensor(data)
#         qlatents = hbq.bit_codes_to_quantized_latent(bit_codes,4)
#         return qlatents
dataset_name='fantasy'
# def get_latent_dataset_and_dataloader(dataset,width=256,height=256,batch_size=8, shuffle=False):
#     bds = dh.BlobDataset(f"data/dbs/quantized_latents_for_{dataset}_{width}x{height}.sqlite3")
#     lds = dh.LatentDataset(bds)
#     ldl = DataLoader(lds,shuffle=shuffle,batch_size=batch_size)
#     return lds,ldl
def get_latent_dataset_and_dataloader(dataset_name,width=256,height=256,batch_size=8, shuffle=False):
    print(f"data/dbs/quantized_latents_for_{dataset_name}_{width}x{height}.sqlite3")
    bds = dh.BlobDataset(f"data/dbs/quantized_latents_for_{dataset_name}_{width}x{height}.sqlite3")
    lds = dh.QuantizedLatentDataset(bds)
    ldl = DataLoader(lds,shuffle=shuffle,batch_size=batch_size)
    return lds,ldl
autoencoder.cpu()
def smoke_test_data_loader_and_autoencoder(dataset):
    lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True,width=width,height=height)
    id_batch, ld_batch, metadata_batch = next(iter(ldl))
    print(ld_batch.shape)
    ipd.display(ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(ld_batch))[0]))

smoke_test_data_loader_and_autoencoder(dataset='fantasy')


## Diffusion Helpers

* Roughtly following the noise schedules and x-prediction strategies of the JiT paper.
  * Generate a logit-normal distribution of noise levels
  * Loss function that measures de-noising effectiveness.
  * Functions to take single steps of denoising
  * Function to start with random noise and loop through denoising steps.

In [ ]:

P_mean, P_std = -0.8,0.8

def sample_t(n, P_mean, P_std, device=None):
    """
    Returns a logit-normal distribution. It's 
    the rectified-flow-world's version of the same 
    idea of EDM's log-normal σ-sampling.

    Determines how much of your compute budget gets spent 
    on which regions of the noise-level spectrum.
    """
    z = torch.randn(n, device=device) * P_std + P_mean
    return torch.sigmoid(z)

def flow_matching_loss(net, x, P_mean, P_std, t_eps, noise_scale=1.0):
    """
    A loss function that says whether a model is good at denoising.
    """
    t = sample_t(x.size(0), P_mean, P_std, device=x.device)
    t = t.view(-1, *([1] * (x.ndim - 1)))          # broadcast over (C,H,W)
    e = torch.randn_like(x) * noise_scale
    z = t * x + (1 - t) * e
    v_target = (x - z) / (1 - t).clamp_min(t_eps)
    x_pred = net(z, t.flatten())
    v_pred = (x_pred - z) / (1 - t).clamp_min(t_eps)
    return ((v_target - v_pred) ** 2).mean()

@torch.no_grad()
def forward_sample(net, z, t, t_eps):
    """ Take a step in a denoising direction. """
    x_pred = net(z, t.flatten())
    v_pred = (x_pred - z) / (1 - t).clamp_min(t_eps)
    return v_pred, x_pred

@torch.no_grad()
def euler_step(net, z, t, t_next, t_eps):
    """ https://en.wikipedia.org/wiki/Euler_method """
    v_pred, x_pred = forward_sample(net, z, t, t_eps)
    return z + (t_next - t) * v_pred, x_pred

@torch.no_grad()
def heun_step(net, z, t, t_next, t_eps):
    """ https://en.wikipedia.org/wiki/Heun%27s_method """
    v_t, x_pred = forward_sample(net, z, t, t_eps)
    z_euler = z + (t_next - t) * v_t
    v_t_next, x_pred_next = forward_sample(net, z_euler, t_next, t_eps)
    v_avg = 0.5 * (v_t + v_t_next)
    return z + (t_next - t) * v_avg, (x_pred + x_pred_next)/2


@torch.no_grad()
def generate(net, batch_size, latent_dim, grid_size, steps, t_eps,
             method="heun", noise_scale=1.0, device="cuda",
             save_steps=False):
    """ Generate an image from pure noise by taking multiple steps. """
    z = noise_scale * torch.randn(batch_size, latent_dim, grid_size, grid_size, device=device)
    t_schedule = torch.linspace(0.0, 1.0, steps + 1, device=device)   # (steps+1,) plain values
    stepper = heun_step if method == "heun" else euler_step
    saved_steps=[]
    for i in range(steps - 1):
        z,x_pred = stepper(net, z, t_schedule[i], t_schedule[i + 1], t_eps)
        saved_steps.append(x_pred.cpu().detach())
    result,x_pred = euler_step(net, z, t_schedule[-2], t_schedule[-1], t_eps)
    saved_steps.append(result)
    return (result,saved_steps) if (save_steps) else result

@torch.no_grad()
def generate(net, batch_size, latent_dim, grid_size, steps, t_eps,
             method="heun", noise_scale=1.0, device="cuda",
             churn_frac=0.0, t_churn_min=0.0, t_churn_max=0.9, noise_mix=0,
             save_steps=False):
    """ Generate an image from pure noise by taking multiple steps. """
    z = noise_scale * torch.randn(batch_size, latent_dim, grid_size, grid_size, device=device)
    t_schedule = torch.linspace(0.0, 1.0, steps + 1, device=device)
    stepper = heun_step if method == "heun" else euler_step
    saved_steps = []
    for i in range(steps - 1):
        t_cur, t_next = t_schedule[i], t_schedule[i + 1]
        z, x_pred = stepper(net, z, t_cur, t_next, t_eps)

        if churn_frac > 0 and t_churn_min <= t_next.item() <= t_churn_max:
            t_back = (t_next - churn_frac * (t_next - t_cur)).clamp_min(t_eps)

            # what noise is *implied* by the current z and the model's own x_pred --
            # reconstructing this lets us preserve most of the existing trajectory
            e_implied = (z - t_next * x_pred) / (1 - t_next).clamp_min(t_eps)
            e_fresh = torch.randn_like(z) * noise_scale

            # noise_mix=0 -> exact backward move, no new randomness at all
            # noise_mix=1 -> full resample (this was the *only* behavior available before)
            e_mixed = math.sqrt(1 - noise_mix**2) * e_implied + noise_mix * e_fresh

            z = t_back * x_pred + (1 - t_back) * e_mixed

        saved_steps.append(x_pred.cpu().detach())
    result, x_pred = euler_step(net, z, t_schedule[-2], t_schedule[-1], t_eps)
    saved_steps.append(result)
    return (result, saved_steps) if save_steps else result

In [ ]:
sample_t(100,-0.8,0.8)

## Create a model to train

In [ ]:
t_eps = 0.001
latent_dim=16
torch.random.manual_seed(42)

lj = MinimalSpatialViT(grid_size=grid_size,
                        latent_dim=latent_dim,
                        #theta=1000.0
                       ).to(device)
device='cuda'
lj = lj.to(device)

autoencoder.to('cpu')
result = generate(lj,3,steps=50, t_eps=t_eps,grid_size=grid_size,latent_dim=latent_dim)
hbq_misc.image_helpers.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(result.cpu()))[0].cpu())


In [ ]:
#lj = ch.create_from_checkpoint(MinimalSpatialViT, "checkpoints/nice_2026-09-13_19-48-58_fantasy_cs336_latent_jit.pth").to('cuda')

In [ ]:
!ls -lrt checkpoints/

In [ ]:
# if good_latent_xpred_diffusion_cp: 
#   ch.load_checkpoint(lj,path=good_latent_xpred_diffusion_cp,optimizer=None)

#ch.load_checkpoint(lj,None,"checkpoints/nice_2026-09-13_19-48-58_fantasy_cs336_latent_jit.pth")
#result = generate(lj,3,steps=90, t_eps=t_eps,grid_size=32,latent_dim=latent_dim, noise_scale=0.8)
ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(result.cpu()))[0].cpu())


In [ ]:
dataset='fantasy'
#width,height=384,384
# 6 worked for 256x256
lds,data_loader_train = get_latent_dataset_and_dataloader(dataset,width=width,height=height,shuffle=True, batch_size=2)

In [ ]:
def smoke_test_LatentDenoiser_model(lj):
    lj.eval()
    lj.cuda()
    x = torch.randn(3, latent_dim, grid_size, grid_size).cuda()
    labels = torch.zeros(3, dtype=torch.long).cuda()
    with torch.no_grad():
        P_mean = -0.8
        P_std = 0.8
        t_eps = 0.05
        loss = flow_matching_loss(lj, x, P_mean, P_std, t_eps, noise_scale=1.0)
    print(loss)
smoke_test_LatentDenoiser_model(lj)

In [ ]:
device='cuda'
lj = torch.compile(lj)
lj = lj.to(device)
optimizer = torch.optim.AdamW(lj.parameters(), lr=1e-4, betas=(0.9, 0.95))  # was 3
n_params = sum(p.numel() for p in lj.parameters() if p.requires_grad)
print("Model =", lj, "Number of trainable parameters: {:.6f}M".format(n_params / 1e6))

## ML FLow Logging

In [ ]:
# import mlflow
# def restart_mlflow_run(mlflow,mlflow_experiment,mlflow_run_name, mlflow_run_id, loggable_params):
#     try:
#         mlflow.end_run()
#     except:
#         pass
#     mlflow.set_experiment(mlflow_experiment)
#     try:
#         mlflow.start_run(run_id=mlflow_run_id, run_name=mlflow_run_name)
#     except Exception as e:
#         print(f"⚠️ Run {mlflow_run_id} not found/corrupted: {e}. Starting fresh.")    
#         # 2. Start a new run if the old one fails
#         new_run = mlflow.start_run(run_name=mlflow_run_name)
#     for k,v in loggable_params.items():
#         mlflow.log_param(k,v)

# def log_to_mlflow(mlflow,elapsed_seconds,samples_processed,loss,sigma_bins,sigma_bin_loss_ema,samples_per_second):
#     # ------------------------------------------------------------
#     # 🔥 MLFLOW: LOG METRICS
#     # View with mlflow ui
#     # ------------------------------------------------------------
#     print("ES is ",elapsed_seconds)
#     if elapsed_seconds > 60:
#         # just screws up mlflow graphs if you log too early or too frequently or with scales like epochs
#         mlflow_step = int(elapsed_seconds / 60)
#         mlflow.log_metric("loss", float(loss), step=mlflow_step)
#         mlflow.log_metric("samples_per_second", samples_per_second, step=mlflow_step)
#         if sigma_bins:
#             for k,v in sigma_bin_loss_ema.items():
#                 kname = f"loss_bin_{k}_{sigma_bins[k]:.3f}" if k < len(sigma_bins) else f"loss_bin_{k}_100.000"
#                 mlflow.log_metric(kname, v, step=mlflow_step)


## Training Loop

In [ ]:


def show_some(lj, title="", output_file="outputs/tmp.html", grid_size=grid_size):
    with torch.no_grad():
        device = next(lj.parameters()).device
        lj.to('cuda')
        #imgs = model.generate(labels[0:3].cuda()).cpu()
        imgs = generate(lj,latent_dim=latent_dim,grid_size=grid_size,steps=50,t_eps=0.05, batch_size=3).cpu()
        lj.to(device)
        decoded =autoencoder.decode(autoencoder.post_quant(imgs))
        pils = [ih.tensor_to_pil(i) for i in decoded]
        h = ih.html_for_images(pils,title=title)
        if output_file:
            with open(output_file,"a") as f:
                f.write(h)
        ipd.display(ipd.HTML(h))

def train_some(model, dataloader, ema, run_name):

    ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

    model_name = model.__class__.__name__
    if hasattr(model, '_orig_mod'):
        model_name = f"{model._orig_mod.__class__.__name__} optimized"

    if run_name is None:
        run_name = f"{model_name}_{ts}"
    mlflow_helper = mh.MLFlowHelper(dataset, run_name=run_name, loggable_params=model.config)
    best_loss = 1_000_000
    
    model.train()
    t0 = time.time()
    counter = 0
    display_time_interval = 10
    next_display_time = time.time() + display_time_interval
    num_samples = len(dataloader.dataset)
    num_batches = len(dataloader)
    
    #ema.get_model().config = {"TODO":"populate config"}
    output_file = f"outputs/{run_name}_{ts}.html"
    checkpoint_path = f"checkpoints/{ts}_{run_name}.pth"
    for epoch in range(1_000_000):
        for step, batches in enumerate(dataloader):
            id_batch, images, metadata_batch = batches
            labels = torch.zeros(images.shape[0], dtype=torch.long).cuda()
            images = images.cuda()
            labels = labels.cuda()
            optimizer.zero_grad()
            #loss = model(images, labels)
            loss = flow_matching_loss(lj, images, -0.8, 0.8, 0.05)
            loss.backward()
            optimizer.step()
            counter += images.shape[0]
            ema.update(model._orig_mod)
            if time.time() > next_display_time:
                elapsed_time = time.time()-t0

                metrics = {
                    "elapsed_seconds":elapsed_time,
                    "samples_processed":counter,
                    "loss":loss,
                    "samples_per_second": counter / elapsed_time,
                }
                mlflow_step = elapsed_time // 60
                mlflow_helper.log_to_mlflow(mlflow_step, metrics)
                ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
                #model.config = {"TODO":"populate config"}
                if loss.item() < best_loss:
                    cp_path = f"checkpoints/{model_name}_{ts}.pth"
                    ch.save_checkpoint(model._orig_mod, optimizer=optimizer)
                    ch.save_checkpoint(ema.get_model(), None)
                    best_loss = loss.item()
                status = f"{model_name} epoch {epoch}.{step*100//num_batches:02d}, rows {counter}, tm {elapsed_time}, loss {loss.item()}"
                ipd.clear_output(wait=True)
                model.eval()
                show_some(model, status, output_file)
                show_some(ema.get_model(), f"EMA {model_name} {counter}", output_file)
                model.train()
                display_time_interval = min(display_time_interval * 1.2,60*15)
                next_display_time = time.time() + display_time_interval
                
ema = hbq_misc.ema_helper.EDMEMAHelper(lj._orig_mod, 
                step=1, batch_size=data_loader_train.batch_size, 
                ema_halflife_kimg=len(data_loader_train.dataset)/1000
                )



## Train

In [ ]:
# import mlflow
# from mlflow.tracking import MlflowClient

# def emergency_mlflow_reset(new_db_path="mlflow_recovered.db"):
#     print("⚠️ Purging stale MLflow connections and tracking registries...")
    
#     # 1. Clear out internal SQLAlchemy store registries completely
#     try:
#         mlflow.tracking._tracking_service.utils._tracking_store_registry.stores.clear()
#     except Exception:
#         pass

#     # 2. Point to a brand new SQLite destination file to force schema generation
#     new_uri = f"sqlite:///{new_db_path}"
#     mlflow.set_tracking_uri(new_uri)
    
#     # 3. Instantiate a fresh client isolated from previous file handles
#     client = MlflowClient(tracking_uri=new_uri)
    
#     # 4. Spin up a new experiment and run structure 
#     # (Since the old run uuid cannot be mapped to the deleted schema)
#     try:
#         # Gracefully sever ties with the active context if it exists
#         mlflow.end_run() 
#     except Exception:
#         pass
        
#     exp_id = mlflow.set_experiment("recovered_runs")
#     new_run = mlflow.start_run()
    
#     print(f"🚀 MLflow successfully bound to clean backend! New Run ID: {new_run.info.run_id}")
#     return new_run

# # Execute the reset right before your next logging step
# emergency_mlflow_reset()


In [ ]:
if train := True:
    try:
        train_some(lj,data_loader_train, ema, run_name=None)
    except KeyboardInterrupt as e:
        print("done")
        lj.zero_grad(set_to_none=True)

# Runs for hours, updating the image below, until the image above gradually becomes more like the ones below.

In [ ]:
lj.config

In [ ]:
if train := True:
    try:
        train_some(lj,data_loader_train, ema, run_name=f"{dataset}_finetune_attempt")
    except KeyboardInterrupt as e:
        print("done")
        lj.zero_grad(set_to_none=True)

# Runs for hours, updating the image below, until the image above gradually becomes more like the ones below.

In [ ]:
1/0

## Show some results

In [ ]:
lj.eval()
labels = torch.zeros(3, dtype=torch.long).cuda()
show_some(lj, "cs336_with_adaln", None)
show_some(ema.get_model(), "cs336_with_adaln ema", None)

In [ ]:
ts = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
checkpoint_path=f"checkpoints/MinimalSpatialViT_{width}x{height}_{ts}_{dataset}_cs336_latent_jit.pth"
ch.save_checkpoint(lj._orig_mod,checkpoint_path,optimizer)
ch.save_checkpoint(ema.get_model(), f"{checkpoint_path}.ema.pth")



In [ ]:
lj.eval()
for i in range(3):
    labels = torch.zeros(3, dtype=torch.long).cuda()
    show_some(lj, "v6 cs336", "outputs/done_v5_cs336.html")
for i in range(3):
    labels = torch.zeros(3, dtype=torch.long).cuda()
    show_some(ema.get_model(), "v6 cs336 ema", "outputs/done_v5_cs336.html")

## Debug showing different perspectives on the denoising process

In [ ]:
lj = ch.create_from_checkpoint(MinimalSpatialViT,'checkpoints/MinimalSpatialViT_2026-09-14_08-46-14_fantasy_cs336_latent_jit.pth.ema.pth')

In [ ]:
lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = next(iter(ldl))[1].to(device)
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x.to('cpu')))
ipd.display(ih.tensor_to_pil(decoded[0]))
for t_val in torch.arange(0.0,1.0,0.2):
    t = torch.full((x.size(0),), t_val, device=device)
    e = torch.randn_like(x)
    z = t.view(-1,1,1,1) * x + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z.to('cpu')))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred.cpu()))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

## Totally unsupervised generation

In [ ]:
lj.to('cuda')
result,steps = generate(lj,1,lj.config["latent_dim"],lj.config["grid_size"],90,0.001,noise_scale=0.95,save_steps=True)
final_img = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(result.cpu()))[0])
imgs = []
for step in steps[::]:
    decoded = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(step.cpu()))[0])
    imgs.append(decoded)
ipd.display(ipd.HTML(ih.html_for_images(imgs[::5],"Unsupervised Generation Steps")))
ipd.display(final_img)


# Churn frac

In [ ]:

@torch.no_grad()
def generate(net, batch_size, latent_dim, grid_size, steps, t_eps,
             method="heun", noise_scale=1.0, device="cuda",
             churn_frac=0.0, t_churn_min=0.0, t_churn_max=0.9, noise_mix=0,
             save_steps=False):
    """ Generate an image from pure noise by taking multiple steps. """
    z = noise_scale * torch.randn(batch_size, latent_dim, grid_size, grid_size, device=device)
    t_schedule = torch.linspace(0.0, 1.0, steps + 1, device=device)
    stepper = heun_step if method == "heun" else euler_step
    saved_steps = []
    for i in range(steps - 1):
        t_cur, t_next = t_schedule[i], t_schedule[i + 1]
        z, x_pred = stepper(net, z, t_cur, t_next, t_eps)


        if churn_frac > 0 and t_churn_min <= t_next.item() <= t_churn_max:
            sigma_cur = (1 - t_next) * noise_scale
            sigma_extra = churn_frac * sigma_cur                         # relative to current noise, not a fixed tiny dt
            sigma_new = min((sigma_cur**2 + sigma_extra**2) ** 0.5, noise_scale)   # hard clamp -- can't exceed the valid max
            if sigma_new < noise_scale:
                t_back = 1 - sigma_new / noise_scale
                z = z + sigma_extra * torch.randn_like(z)
                #print(sigma_new, t_schedule[i + 1], t_back)
                t_schedule[i + 1] = (t_back + t_schedule[i+1])/2

        # if churn_frac > 0 and t_churn_min <= t_next.item() <= t_churn_max:
        #     sigma_cur = (1 - t_next) * noise_scale                         # current effective noise std in z
        #     sigma_extra = churn_frac * (t_next - t_cur) * noise_scale       # how much extra to add
        #     z = z + sigma_extra * torch.randn_like(z)                       # pure addition -- x_pred never enters
        #     sigma_new = math.sqrt(sigma_cur**2 + sigma_extra**2)
        #     t_back = 1 - sigma_new / noise_scale
        #     t_schedule[i + 1] = t_back
        #     print(t_back)

        # if churn_frac > 0 and t_churn_min <= t_next.item() <= t_churn_max:
        #     t_back = (t_next - churn_frac * (t_next - t_cur)).clamp_min(t_eps)

        #     # what noise is *implied* by the current z and the model's own x_pred --
        #     # reconstructing this lets us preserve most of the existing trajectory
        #     e_implied = (z - t_next * x_pred) / (1 - t_next).clamp_min(t_eps)
        #     e_fresh = torch.randn_like(z) * noise_scale

        #     # noise_mix=0 -> exact backward move, no new randomness at all
        #     # noise_mix=1 -> full resample (this was the *only* behavior available before)
        #     e_mixed = math.sqrt(1 - noise_mix**2) * e_implied + noise_mix * e_fresh

        #     z = t_back * x_pred + (1 - t_back) * e_mixed

        saved_steps.append(x_pred.cpu().detach())
    result, x_pred = euler_step(net, z, t_schedule[-2], t_schedule[-1], t_eps)
    saved_steps.append(result)
    return (result, saved_steps) if save_steps else result

In [ ]:
lj.to('cuda')
#torch.manual_seed(4)
result,steps = generate(lj,1,lj.config["latent_dim"],lj.config["grid_size"],t_eps=0.001,steps=40,
                        churn_frac=0.3,noise_mix=0.2,save_steps=True,method="euler")
final_img = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(result.cpu()))[0])
imgs = []
for step in steps[::]:
    decoded = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(step.cpu()))[0])
    imgs.append(decoded)
ipd.display(ipd.HTML(ih.html_for_images(imgs[::5],"")))
ipd.display(final_img)


# Other loop idea

In [ ]:
x.shape

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.randn(1,16,32,32).to(device)
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x.to('cpu')))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x)
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z.to('cpu')))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred.cpu()))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,32,32).to(device)
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x.to('cpu')))
ipd.display(ih.tensor_to_pil(decoded[0]))
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    e = torch.randn_like(x)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z.to('cpu')))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred.cpu()))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,32,32).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x.to('cpu')))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x) * 0.75
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z.to('cpu')))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred.cpu()))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))

In [ ]:
#lds,ldl = get_latent_dataset_and_dataloader(dataset,shuffle=True, batch_size=1)
x = torch.rand(1,16,32,32).to(device)*2-1
x_pred = x.clone()
print("Original")
decoded = autoencoder.decode(autoencoder.post_quant(x.to('cpu')))
ipd.display(ih.tensor_to_pil(decoded[0]))
e = torch.randn_like(x) * 0.75
for t_val in torch.arange(0.0,1.0,0.1):
    t = torch.full((x.size(0),), t_val, device=device)
    z = t.view(-1,1,1,1) * x_pred + (1 - t.view(-1,1,1,1)) * e
    with torch.no_grad():
        x_pred = lj(z, t)

    noised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(z.to('cpu')))[0])
    denoised = ih.tensor_to_pil(autoencoder.decode(autoencoder.post_quant(x_pred.cpu()))[0])
    print(t_val)
    ipd.display(ipd.HTML(ih.html_for_images([noised,denoised],f"time {t}")))